## Aviso de versión y resultados vigentes

Este notebook contiene celdas y explicaciones de iteraciones anteriores. Para este corte del proyecto excluimos 2020 de todo análisis y resultado; no uses cifras anteriores que aún aparezcan en celdas o comentarios como resultados finales. Los resultados vigentes se recalcularon con los scripts de la raíz del proyecto y se documentan en Procesos_revisado.docx, README.md y los archivos de resultados_eda/, resultados_dask/ y resultados_modelo_v2/.

La base Parquet completa sin 2020 tiene 24.171.385 filas, 61 variables y 678.550.973 bytes. El modelado usa 23.634.669 vuelos etiquetados. Para AWS/Grafana, la carga real no se verificó; no ejecutes celdas de carga con credenciales sin revisar el destino y subir únicamente agregados analíticos.


> **Decisión metodológica:** 2020 se excluye de todo análisis, modelo, evaluación y visualización. El CSV original se conserva intacto. Los resultados guardados en este notebook se generaron antes de aplicar el filtro y son obsoletos; no deben citarse. Reejecute el flujo con `Year <> 2020` antes de usar resultados. Consulte `DECISION_EXCLUSION_2020.md`.

# 1. ANALISIS EXPLORATORIO DE LOS DATOS

Importamos las librerías necesarias para realizar el análisis de los datos.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

Combinamos todas las bases de datos en una sola para facilitar la manipulación y el análisis de la información.

In [ ]:

# Lista con los nombres de tus archivos
archivos = [
    "Combined_Flights_2018.csv",
    "Combined_Flights_2019.csv",
    "Combined_Flights_2021.csv",
    "Combined_Flights_2022.csv",
]

# Leer y concatenar todos en un solo DataFrame
df_list = [pd.read_csv(file) for file in archivos]
df_flights = pd.concat(df_list, ignore_index=True)
df_flights.head(10)  # Mostrar las primeras 10 filas para verificar la carga de datos

Liberamos la lista de DataFrames originales. Tras el `concat`, `df_flights` es una copia independiente, y mantener `df_list` duplicaría el uso de memoria.

In [ ]:
import gc

if "df_list" in globals():
    del df_list
gc.collect()

identificamos cada una de las columnas presentes en el Dataframe

In [ ]:
df_flights.columns  # Mostrar los nombres de las columnas del DataFrame

Identificamos el tipo de variable por cada columna

In [ ]:
df_flights.info()  # Mostrar información del DataFrame para verificar tipos de datos y valores nulos

identificamos la estructura del Dataframe

In [ ]:
df_flights.shape # Mostrar el número de filas y columnas del DataFrame

Se evidencia la existencia de 29.193.782 registros de vuelos durante el periodo comprendido entre enero de 2018 y julio de 2022 (rango que se verifica más adelante con la fecha mínima y máxima), con 61 columnas que describen cada vuelo.

Buscamos si hay valores nulos

In [ ]:
df_flights.isna().sum()  # Contar valores nulos en cada columna

Buscamos si existen registros duplicados

In [ ]:
df_flights.duplicated().sum()  # Contar filas duplicadas en el DataFrame

In [ ]:

cols = ["ArrDel15", "ArrTimeBlk"]

# 1) Cuántos nulos hay y qué porcentaje del total
print(df_flights[cols].isna().sum())
print((df_flights[cols].isna().mean() * 100).round(2))

# 2) ¿Los nulos de ambas columnas son los mismos registros?
mask_arr = df_flights["ArrDel15"].isna()
mask_blk = df_flights["ArrTimeBlk"].isna()
print("Nulos en ambas:", (mask_arr & mask_blk).sum())
print("Solo ArrDel15:", (mask_arr & ~mask_blk).sum())
print("Solo ArrTimeBlk:", (~mask_arr & mask_blk).sum())

# 3) ¿Coinciden con cancelados y desviados?
print(pd.crosstab(
    df_flights["Cancelled"], df_flights["Diverted"],
    values=mask_arr, aggfunc="sum", margins=True
))
print(df_flights.loc[mask_arr, ["Cancelled", "Diverted"]].value_counts(dropna=False))

# 4) Nulos "sin explicación": no cancelado y no desviado
resto = mask_arr & (df_flights["Cancelled"] == 0) & (df_flights["Diverted"] == 0)
print("Nulos sin explicación operativa:", resto.sum())

# 5) Distribución por año (2020 debería destacar si es el caso)
print(df_flights.loc[mask_arr].groupby("Year").size())

Existen columnas que pueden ser optimizadas para reducir el uso de memoria por lo que convertiremos el tipo de dato a `Category`

In [ ]:
for c in ["Airline", "Origin", "Dest", "OriginState", "DestState"]:
    df_flights[c] = df_flights[c].astype("category")
df_flights["FlightDate"] = pd.to_datetime(df_flights["FlightDate"])

In [ ]:
df_flights.info()  # Verificar los tipos de datos después de la conversión

Verificamos el rango de fechas real del conjunto de datos y la cantidad de días distintos.

In [ ]:
print("Fecha mínima:", df_flights["FlightDate"].min())
print("Fecha máxima:", df_flights["FlightDate"].max())
print("Días distintos:", df_flights["FlightDate"].nunique())

### Clasificación de las variables

In [ ]:
resumen = pd.DataFrame({
    "tipo": df_flights.dtypes,
    "valores_distintos": df_flights.nunique(),
})
print(resumen)

In [ ]:
pd.set_option("display.max_rows", None)
print(resumen.sort_values("valores_distintos"))

### Nulos de DepDel15

Verificamos si los valores vacíos de `DepDel15` corresponden a vuelos cancelados (nulos estructurales: un vuelo que no despega no tiene retraso de salida) o a errores de captura. Si al aislar los vuelos no cancelados los nulos desaparecen o son marginales, se justifica no imputarlos, para no distorsionar la distribución real con falsos vuelos operados.

Nota: `DepDel15` es la base de la variable objetivo de este análisis (la tasa de retraso por aeropuerto-día). Por lo tanto, en un eventual modelo, la tasa del mismo día no puede usarse como variable predictora (fuga de información).

In [ ]:
print("Nulos totales en DepDel15:", df_flights["DepDel15"].isna().sum())
print("Nulos en vuelos NO cancelados:", df_flights.loc[~df_flights["Cancelled"], "DepDel15"].isna().sum())

Casi todos los nulos corresponden a cancelaciones: de 763,084 nulos, solo 1,113 ocurren en vuelos no cancelados (0.004 % del total). Son marginales y no se imputan.

### Verificación: ¿un vuelo cancelado puede tener retraso de salida?
Un vuelo cancelado normalmente no despega, así que no debería tener un valor en `DepDel15`. Si lo tiene, ese vuelo se contaría como si hubiera salido y la tasa de retraso quedaría distorsionada. Aquí solo contamos, sin modificar nada.

In [ ]:
print("Cancelados que tienen valor en DepDel15:",
      df_flights.loc[df_flights["Cancelled"], "DepDel15"].notna().sum())

### Construcción de la tabla aeropuerto-día
Cambiamos la unidad de análisis: pasamos del vuelo individual a una métrica agregada por **aeropuerto de origen y fecha**. Cada fila resume un día de operación en un aeropuerto, y reduce unos 29 millones de filas a unas 580 mil. Esa tabla es la base de lo que luego se persiste en AWS y se visualiza en Grafana.

Decisiones de cálculo:
- El retraso de salida de los vuelos cancelados se deja vacío: un vuelo que no despegó no cuenta ni como puntual ni como retrasado. Es un supuesto, porque algunos cancelados traen valor en `DepDel15` (por ejemplo, vuelos que pudieron salir del puerta y regresar).
- `tasa_retraso` se calcula sobre los vuelos con dato (`count`), que incluyen los desviados porque sí despegaron.
- `tasa_cancelacion` se calcula sobre todos los vuelos programados (`size`).

In [ ]:
# 1. Copia pequeña con solo las 4 columnas necesarias
base = df_flights[["Origin", "FlightDate", "DepDel15", "Cancelled"]].copy()

# 2. Un vuelo cancelado no tiene retraso de salida: se deja vacío
base.loc[base["Cancelled"], "DepDel15"] = np.nan

# 3. Una fila por aeropuerto y día
aero_dia = base.groupby(["Origin", "FlightDate"], observed=True).agg(
    vuelos_programados=("DepDel15", "size"),   # todas las filas del grupo
    vuelos_con_dato=("DepDel15", "count"),     # filas con valor (no vacías)
    vuelos_retrasados=("DepDel15", "sum"),     # cuántos 1 hay = vuelos retrasados
    cancelados=("Cancelled", "sum"),           # cuántos True hay
).reset_index()

# 4. Tasas
aero_dia["tasa_retraso"] = aero_dia["vuelos_retrasados"] / aero_dia["vuelos_con_dato"]
aero_dia["tasa_cancelacion"] = aero_dia["cancelados"] / aero_dia["vuelos_programados"]

# 5. Chequeos: ningún vuelo debe perderse al agrupar
assert aero_dia["vuelos_programados"].sum() == len(df_flights)
print(aero_dia.shape)
print("Total de vuelos en aero_dia:", aero_dia["vuelos_programados"].sum())
print("Aeropuerto-día sin ningún vuelo con dato (todos cancelados):", aero_dia["tasa_retraso"].isna().sum())
print(aero_dia.head())
print(aero_dia["vuelos_programados"].describe())

### Tasa de retraso de la red: dos formas de calcularla
Para juzgar si un aeropuerto retrasa mucho o poco, necesitamos un valor de comparación. Lo calculamos de dos maneras. En la primera, sumamos todos los vuelos de todos los aeropuertos y días (la tasa real de la red). En la segunda, promediamos las tasas de cada fila, de modo que un día con 3 vuelos pesa igual que un día con 1,000. Si los dos resultados difieren, queda demostrado por qué hay que ponderar por volumen.


In [ ]:
tasa_nacional = aero_dia["vuelos_retrasados"].sum() / aero_dia["vuelos_con_dato"].sum()
promedio_simple = aero_dia["tasa_retraso"].mean()

print("Tasa nacional (ponderada por vuelos):", round(tasa_nacional, 4))
print("Promedio simple de las tasas:        ", round(promedio_simple, 4))

### El problema de los días con pocos vuelos
Si un aeropuerto tiene 3 vuelos en un día, su tasa de retraso solo puede ser 0 %, 33 %, 67 % o 100 %, y un solo vuelo la cambia 33 puntos. Eso es ruido del azar, no información sobre el aeropuerto. Para medir cuánto ruido hay, agrupamos las filas por rango de volumen y miramos qué tanto varía la tasa dentro de cada rango. Si la variación baja al subir el volumen, queda demostrado que las filas pequeñas son poco confiables y que conviene filtrarlas.


In [ ]:
# 1. Poner cada fila en un rango según su número de vuelos con dato
aero_dia["rango_volumen"] = pd.cut(
    aero_dia["vuelos_con_dato"],
    bins=[-1, 0, 5, 10, 30, 100, 300, 2000],
)

# 2. Resumir cada rango
resumen_vol = aero_dia.groupby("rango_volumen", observed=True).agg(
    filas=("tasa_retraso", "size"),
    vuelos=("vuelos_con_dato", "sum"),
    tasa_media=("tasa_retraso", "mean"),
    desviacion=("tasa_retraso", "std"),
)

# 3. Qué porcentaje de las filas y de los vuelos cae en cada rango
resumen_vol["pct_filas"] = (resumen_vol["filas"] / resumen_vol["filas"].sum() * 100).round(1)
resumen_vol["pct_vuelos"] = (resumen_vol["vuelos"] / resumen_vol["vuelos"].sum() * 100).round(1)

print(resumen_vol.round(3))

### Filtro de volumen mínimo
A partir de la tabla anterior fijamos el mínimo de vuelos que debe tener un aeropuerto-día para entrar al análisis de la tasa de retraso. Se crea una tabla nueva, `aero_dia_f`, y la tabla original no se modifica. Se usa `>` 30 para que el filtro coincida con el borde del rango `(30, 100]`. Anotamos cuántas filas se descartan y qué porcentaje de los vuelos se conserva, como evidencia de la decisión.

Limitación: el filtro deja fuera a los aeropuertos pequeños, así que el análisis queda sesgado hacia aeropuertos medianos y grandes.

In [ ]:
MIN_VUELOS = 30   # se conservan los aeropuerto-día con MÁS de 30 vuelos con dato

aero_dia_f = aero_dia[aero_dia["vuelos_con_dato"] > MIN_VUELOS].copy()

print("Filas antes:  ", len(aero_dia))
print("Filas después:", len(aero_dia_f))
print("% de filas descartadas:", round((1 - len(aero_dia_f) / len(aero_dia)) * 100, 1))
print("% de vuelos que se conserva:",
      round(aero_dia_f["vuelos_con_dato"].sum() / aero_dia["vuelos_con_dato"].sum() * 100, 1))
print("Filas con tasa_retraso vacía:", aero_dia_f["tasa_retraso"].isna().sum())

### Guardado de las tablas
Guardamos las tablas agregadas en formato Parquet (comprimido y rápido) para retomar el trabajo sin recalcular. Estas tablas serán la entrada del flujo de Prefect y de lo que se sube a AWS. Requiere `pip install pyarrow`.

In [ ]:
aero_dia.drop(columns="rango_volumen").to_parquet("aero_dia.parquet")
aero_dia_f.drop(columns="rango_volumen").to_parquet("aero_dia_filtrado.parquet")
print("Guardado.")

### Evidencia de tamaño de los datos
Registramos el tamaño de los datos originales, de la tabla en memoria y de las tablas agregadas. Sirve para el informe: el dataset original supera los 1.5 GB y el procesamiento lo reduce a una tabla pequeña, que es la única que se sube a AWS.

In [ ]:
from pathlib import Path

total_gb = sum(Path(f).stat().st_size for f in archivos) / 1e9
print(f"Archivos originales en disco: {total_gb:.2f} GB")
print("df_flights en memoria:", round(df_flights.memory_usage(deep=True).sum() / 1e9, 1), "GB")

for nombre in ["aero_dia.parquet", "aero_dia_filtrado.parquet"]:
    print(nombre, round(Path(nombre).stat().st_size / 1e6, 1), "MB")